# Optimization

Find the Pareto-optimal combinations of scenarios across all the stands of a project. The goal is to find the one scenario picked per stand, chosen so that no other combination is better in every target variable at once.

Stand areas come from the project's own `inputs/stand_data.json`. If it carries no area for one of the run's stands, this notebook raises rather than assuming equal areas.

In [ ]:
%matplotlib inline
import analysis.optimization.core as opti_core
import susi.io.load_output_data as load_output
import susi.io.utils as io_utils
from analysis.notebooks.components import (
    folder_selection,
    optimization,
    variable_selection,
)
from analysis.optimization.stand_areas import stand_areas_for_run
from susi.io import project_layout
from susi.io.app_settings import AppSettings

## Project and run selection
The optimization works over the folder holding one subfolder per stand, which is a *run* of a project (`projects/<project>/outputs/<run_id>/`), not the project folder itself. Narrow down to it as follows:

**projects root** (change only if your projects live outside the default root)

|

**project**

|

**run**

In [ ]:
folder_browser = folder_selection.build_folder_browser(AppSettings().projects_root)

In [ ]:
projects_root = folder_selection.resolve_start_folder(
    folder_browser, default=AppSettings().projects_root
)
project_dropdown = folder_selection.build_dropdown(projects_root, label="project")

In [ ]:
run_dropdown = folder_selection.build_run_dropdown(project_dropdown.value)

## Alternative: hardcode path
You may also choose the same folder at the following path.
(This is less useful to discover directories, but probably more useful when writing the script to analyze a single project)

In [ ]:
run_dir = project_layout.outputs_dir_for_project(projects_root / project_dropdown.value.name) / run_dropdown.value.name
print(run_dir)

## Stand areas
Every stand is weighted by its area, so that larger stands count for more when the stands are combined into project-level totals.

In [ ]:
stand_areas_ha = stand_areas_for_run(run_dir)
optimization.display_stand_areas(stand_areas_ha)

## Choose target variables
Tick the netcdf variables to optimize over. Three are preselected as a starting point; edit `DEFAULT_VARIABLES` below to change them.

In [ ]:
# The golden test netcdf is used only to read the variable structure of the
# netcdf file (names/shapes/units), not its values.
sample_netcdf_filepath = (
    io_utils.repo_root() / "tests/golden_file_test/golden_susi.nc"
)
all_variables = load_output.list_all_netcdf_variables(sample_netcdf_filepath)

DEFAULT_VARIABLES = [
    load_output.NetcdfVariablePath("/stand/volume"),
    load_output.NetcdfVariablePath("/balance/C/soil_c_balance_co2eq"),
    load_output.NetcdfVariablePath("/balance/N/to_water"),
]

selector = variable_selection.build_selector(
    all_variables, preselected=DEFAULT_VARIABLES
)

## Configure the targets
For each chosen variable, pick how its time/space array collapses to a single number, and whether to **minimize** or **maximize** it. The direction is about the signed value: maximizing a variable that is always negative brings it closer to zero.

In [ ]:
chosen_netcdf_variables = variable_selection.checked_variables(selector, all_variables)

if not chosen_netcdf_variables:
    print("No variables chosen")
else:
    target_spec_widgets = optimization.build_target_spec_widgets(
        list(chosen_netcdf_variables.keys())
    )

## Alternative: targets in code
The cell below reads the choices above and prints the code you would need to get the same result.

So if you copy-paste the code below, the widgets above are no longer needed.

(The widgets are the easier way to explore the variables. Code is probably more useful when writing the script to analyze a single project.)

In [ ]:
target_specs = optimization.target_specs_from_widgets(target_spec_widgets)
print(optimization.format_target_specs_as_code(target_specs))

## Read and pre-prune the data
Reads every stand's netcdf files and reduces them to one area-weighted number per scenario and target variable, then drops the scenarios that are already beaten within their own stand.

The printout says how big the remaining search is. Check it before running the next cell, which is the expensive one.

In [ ]:
prepared = opti_core.prepare_optimization_data(
    target_specs=target_specs,
    run_dirpath=run_dir,
    stand_areas=stand_areas_ha,
)

## Find the Pareto front
The dynamic-programming search itself. This is the slow step, and its cost grows with the number of combinations printed above and with how small `epsilon` is.

`epsilon` controls the precision of the Pareto front: smaller is more precise but slower. `n_random_points` are sampled alongside it, to show in the plots what an arbitrary (non-optimal) combination looks like.

In [ ]:
epsilon = 1e-7
n_random_points = 10000

results = opti_core.solve_optimization(
    prepared=prepared,
    epsilon=epsilon,
    n_random_points=n_random_points,
)

## Results
Each panel plots one pair of target variables: the Pareto front against the random combinations.

In [ ]:
figure = optimization.display_pareto_corner_plot(
    results, labels=list(target_specs.keys())
)